# 01 Baselines

Run the baseline models for one selected horizon and compare metrics.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
import pandas as pd

from src.data.load import load_dataset
from src.data.split import get_test_df
from src.evaluation.diagnostics import ljung_box_diagnostics
from src.evaluation.dm_test import diebold_mariano_test
from src.models.lasso import evaluate_lasso_model, predict_lasso
from src.models.linear import evaluate_linear_model, predict_linear
from src.models.naive import evaluate_naive_model, predict_naive

df = load_dataset()
test_df = get_test_df(df)
horizon = 1

In [ ]:
naive_results = evaluate_naive_model(test_df, horizon)
linear_model, linear_results = evaluate_linear_model(df, horizon)
lasso_model, lasso_results = evaluate_lasso_model(df, horizon)

linear_predictions = predict_linear(linear_model, test_df)
lasso_predictions = predict_lasso(lasso_model, test_df)
naive_predictions = predict_naive(test_df, horizon)

comparison = pd.concat([naive_results, linear_results, lasso_results], ignore_index=True)
comparison

In [ ]:
dm_result = diebold_mariano_test(test_df["target_h1"], linear_predictions, naive_predictions)
diagnostics = ljung_box_diagnostics(test_df["target_h1"] - linear_predictions, lags=[5])
{
    "dm_result": dm_result,
    "ljung_box": diagnostics.to_dict(orient="records"),
}